In [14]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers
import matplotlib.pyplot as plt

# ================= CONFIGURAZIONE =================
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Percorsi (Devono corrispondere a dove l'Encoder ha salvato i file)
INPUT_PATH = "processed_step2_triple_head"
MODEL_PATH = "saved_models_dynamics_lstm"
os.makedirs(MODEL_PATH, exist_ok=True)

# 1. Caricamento Dati
print("Caricamento dataset latenti...")
try:
    train_raw = pd.read_csv(os.path.join(INPUT_PATH, "train_dataset.csv")).values.astype(np.float32)
    val_raw   = pd.read_csv(os.path.join(INPUT_PATH, "val_dataset.csv")).values.astype(np.float32)
    test_raw  = pd.read_csv(os.path.join(INPUT_PATH, "test_dataset.csv")).values.astype(np.float32)
    LATENT_DIM = train_raw.shape[1]
    print(f"✅ Dati caricati! Latent Dim: {LATENT_DIM}")
except FileNotFoundError:
    print("❌ ERRORE: File non trovati. Hai lanciato lo Step 2?")

# 2. Funzione Windowing (Crea le sequenze temporali)
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        xs.append(data[i : i + seq_length])    # Input: da t a t+window
        ys.append(data[i + seq_length])        # Target: t+window+1
    return np.asarray(xs, dtype=np.float32), np.asarray(ys, dtype=np.float32)

# 3. COSTRUTTORE MODELLO (LSTM ROBUSTA)
def build_lstm_model(seq_len, latent_dim, cfg):
    inputs = layers.Input(shape=(seq_len, latent_dim))
    
    # Input Norm: Sicurezza extra anche se i dati sono già buoni
    x = layers.BatchNormalization(name="Input_Norm")(inputs)
    
    # LSTM Layer 1
    x = layers.LSTM(cfg['units1'], return_sequences=True, 
                    kernel_regularizer=regularizers.l2(cfg['l2']))(x)
    if cfg.get('use_bn', True): x = layers.BatchNormalization()(x)
    x = layers.Dropout(cfg['dropout'])(x)
    
    # LSTM Layer 2
    x = layers.LSTM(cfg['units2'], return_sequences=False,
                    kernel_regularizer=regularizers.l2(cfg['l2']))(x)
    if cfg.get('use_bn', True): x = layers.BatchNormalization()(x)
    x = layers.Dropout(cfg['dropout'])(x)
    
    # Head (Dense Intermedio)
    dense_dim = int(cfg.get("dense_units", 128))
    x = layers.Dense(dense_dim, activation="swish")(x)
    
    # Output Layer
    outputs = layers.Dense(latent_dim, activation="linear")(x)
    
    model = models.Model(inputs, outputs, name=cfg['name'])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=cfg['lr']), 
                  loss='mse', metrics=['mae'])
    return model

# 4. Helper per il Training
def train_snippet(cfg):
    print(f"\n🚀 AVVIO CONFIGURAZIONE: {cfg['name']} (Window={cfg['seq_len']})")
    
    # Preparazione Dati
    X_t, y_t = create_sequences(train_raw, cfg['seq_len'])
    X_v, y_v = create_sequences(val_raw, cfg['seq_len'])
    X_test, y_test = create_sequences(test_raw, cfg['seq_len'])
    
    # Build & Callbacks
    model = build_lstm_model(cfg['seq_len'], LATENT_DIM, cfg)
    run_dir = os.path.join(MODEL_PATH, cfg['name'])
    os.makedirs(run_dir, exist_ok=True)
    
    cb = [
        callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
        callbacks.ModelCheckpoint(os.path.join(run_dir, 'best_model.keras'), save_best_only=True)
    ]
    
    # Train
    history = model.fit(
        X_t, y_t, validation_data=(X_v, y_v),
        epochs=100, batch_size=cfg['batch_size'],
        callbacks=cb, verbose=1
    )
    
    # Eval
    scores = model.evaluate(X_test, y_test, verbose=0)
    print(f"\n🏆 RISULTATO FINALE {cfg['name']}:")
    print(f"   Val MSE:  {min(history.history['val_loss']):.5f}")
    print(f"   Test MSE: {scores[0]:.5f}")
    
    return history

Caricamento dataset latenti...
✅ Dati caricati! Latent Dim: 128


In [15]:
# === CONFIG 1: BEST LIKE (Target: Test MSE ~0.27 o meno) ===
cfg_best = {
    'name': 'LSTM_Best_Run',
    'seq_len': 10,
    'units1': 256,
    'units2': 128,
    'dense_units': 128,
    'dropout': 0.2,
    'lr': 1e-3,
    'l2': 1e-5,
    'use_bn': True,
    'batch_size': 64
}

# START
hist_best = train_snippet(cfg_best)


🚀 AVVIO CONFIGURAZIONE: LSTM_Best_Run (Window=10)
Epoch 1/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 23s 35ms/step - loss: 0.6008 - mae: 0.6109 - val_loss: 0.9564 - val_mae: 0.7809 - learning_rate: 0.0010
Epoch 2/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 18s 34ms/step - loss: 0.3957 - mae: 0.4948 - val_loss: 0.5986 - val_mae: 0.6113 - learning_rate: 0.0010
Epoch 3/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 18s 34ms/step - loss: 0.3192 - mae: 0.4443 - val_loss: 0.5034 - val_mae: 0.5619 - learning_rate: 0.0010
Epoch 4/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - loss: 0.2936 - mae: 0.4263 - val_loss: 0.4696 - val_mae: 0.5408 - learning_rate: 0.0010
Epoch 5/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - loss: 0.2786 - mae: 0.4151 - val_loss: 0.4387 - val_mae: 0.5250 - learning_rate: 0.0010
Epoch 6/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - loss: 0.2666 - mae: 0.4061 - val_loss: 0.4147 - val_mae: 0.5098 - learning_rate: 0.0010
Epoch 7/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - loss: 0.2597 - mae: 0.

In [18]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers
import matplotlib.pyplot as plt

# ================= CONFIGURAZIONE =================
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Percorsi (Devono corrispondere a dove l'Encoder ha salvato i file)
INPUT_PATH = "processed_step2_triple_head"
MODEL_PATH = "saved_models_dynamics_lstm"
os.makedirs(MODEL_PATH, exist_ok=True)

# 1. Caricamento Dati
print("Caricamento dataset latenti...")
try:
    train_raw = pd.read_csv(os.path.join(INPUT_PATH, "train_dataset.csv")).values.astype(np.float32)
    val_raw   = pd.read_csv(os.path.join(INPUT_PATH, "val_dataset.csv")).values.astype(np.float32)
    test_raw  = pd.read_csv(os.path.join(INPUT_PATH, "test_dataset.csv")).values.astype(np.float32)
    LATENT_DIM = train_raw.shape[1]
    print(f"✅ Dati caricati! Latent Dim: {LATENT_DIM}")
except FileNotFoundError:
    print("❌ ERRORE: File non trovati. Hai lanciato lo Step 2?")

# 2. Funzione Windowing (Crea le sequenze temporali)
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        xs.append(data[i : i + seq_length])    # Input: da t a t+window
        ys.append(data[i + seq_length])        # Target: t+window+1
    return np.asarray(xs, dtype=np.float32), np.asarray(ys, dtype=np.float32)

# 3. COSTRUTTORE MODELLO (LSTM ROBUSTA)
def build_lstm_model(seq_len, latent_dim, cfg):
    # 1. Input Layer
    inputs = layers.Input(shape=(seq_len, latent_dim))
    
    # Input Norm (Mantienila, fa sempre bene)
    x = layers.BatchNormalization(name="Input_Norm")(inputs)
    
    # 2. LSTM Layers (Il "Cervello" che calcola il movimento)
    x = layers.LSTM(cfg['units1'], return_sequences=True, 
                    kernel_regularizer=regularizers.l2(cfg['l2']))(x)
    if cfg.get('use_bn', True): x = layers.BatchNormalization()(x)
    x = layers.Dropout(cfg['dropout'])(x)
    
    x = layers.LSTM(cfg['units2'], return_sequences=False,
                    kernel_regularizer=regularizers.l2(cfg['l2']))(x)
    if cfg.get('use_bn', True): x = layers.BatchNormalization()(x)
    x = layers.Dropout(cfg['dropout'])(x)
    
    # 3. Head (Elaborazione)
    dense_dim = int(cfg.get("dense_units", 128))
    x = layers.Dense(dense_dim, activation="swish")(x)
    
    # --- QUI CAMBIA LA MAGIA (RESIDUAL CONNECTION) ---
    
    # Invece di predire il futuro, prediciamo il DELTA (la correzione)
    delta = layers.Dense(latent_dim, activation="linear", name="delta_output")(x)
    
    # Recuperiamo l'ultimo frame visto in input (t)
    # Inputs shape: (Batch, 10, 128). Noi vogliamo l'ultimo step temporale -> (Batch, 128)
    last_frame = layers.Lambda(lambda t: t[:, -1, :])(inputs)
    
    # SOMMA: Futuro (t+1) = Presente (t) + Delta
    outputs = layers.Add(name="final_prediction")([last_frame, delta])
    
    # -------------------------------------------------
    
    model = models.Model(inputs, outputs, name=cfg['name'])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=cfg['lr']), 
                  loss='mse', metrics=['mae'])
    return model

# 4. Helper per il Training
def train_snippet(cfg):
    print(f"\n🚀 AVVIO CONFIGURAZIONE: {cfg['name']} (Window={cfg['seq_len']})")
    
    # Preparazione Dati
    X_t, y_t = create_sequences(train_raw, cfg['seq_len'])
    X_v, y_v = create_sequences(val_raw, cfg['seq_len'])
    X_test, y_test = create_sequences(test_raw, cfg['seq_len'])
    
    # Build & Callbacks
    model = build_lstm_model(cfg['seq_len'], LATENT_DIM, cfg)
    run_dir = os.path.join(MODEL_PATH, cfg['name'])
    os.makedirs(run_dir, exist_ok=True)
    
    cb = [
        callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
        callbacks.ModelCheckpoint(os.path.join(run_dir, 'best_model.keras'), save_best_only=True)
    ]
    
    # Train
    history = model.fit(
        X_t, y_t, validation_data=(X_v, y_v),
        epochs=100, batch_size=cfg['batch_size'],
        callbacks=cb, verbose=1
    )
    
    # Eval
    scores = model.evaluate(X_test, y_test, verbose=0)
    print(f"\n🏆 RISULTATO FINALE {cfg['name']}:")
    print(f"   Val MSE:  {min(history.history['val_loss']):.5f}")
    print(f"   Test MSE: {scores[0]:.5f}")
    
    return history

Caricamento dataset latenti...
✅ Dati caricati! Latent Dim: 128


In [19]:
# === CONFIG 1: BEST LIKE (Target: Test MSE ~0.27 o meno) ===
cfg_best = {
    'name': 'LSTM_Best_Run',
    'seq_len': 10,
    'units1': 256,
    'units2': 128,
    'dense_units': 128,
    'dropout': 0.2,
    'lr': 1e-3,
    'l2': 1e-5,
    'use_bn': True,
    'batch_size': 64
}

# START
hist_best = train_snippet(cfg_best)


🚀 AVVIO CONFIGURAZIONE: LSTM_Best_Run (Window=10)
Epoch 1/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 24s 36ms/step - loss: 0.1312 - mae: 0.2785 - val_loss: 0.1414 - val_mae: 0.2946 - learning_rate: 0.0010
Epoch 2/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - loss: 0.0944 - mae: 0.2375 - val_loss: 0.1294 - val_mae: 0.2821 - learning_rate: 0.0010
Epoch 3/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - loss: 0.0777 - mae: 0.2151 - val_loss: 0.1200 - val_mae: 0.2716 - learning_rate: 0.0010
Epoch 4/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 19s 35ms/step - loss: 0.0706 - mae: 0.2050 - val_loss: 0.1115 - val_mae: 0.2613 - learning_rate: 0.0010
Epoch 5/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 19s 35ms/step - loss: 0.0664 - mae: 0.1992 - val_loss: 0.1080 - val_mae: 0.2569 - learning_rate: 0.0010
Epoch 6/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 19s 35ms/step - loss: 0.0636 - mae: 0.1951 - val_loss: 0.1047 - val_mae: 0.2532 - learning_rate: 0.0010
Epoch 7/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 19s 35ms/step - loss: 0.0612 - mae: 0.

In [ ]:
# === CONFIG 2: LIGHT (Modello più leggero) ===
cfg_light = {
    'name': 'LSTM_Light_Run',
    'seq_len': 10,       # Window invariata
    'units1': 128,       # Dimezzati i neuroni
    'units2': 64,        # Dimezzati i neuroni
    'dense_units': 64,   # Dimezzata la testa
    'dropout': 0.1,      # Meno dropout (rete piccola overfitta meno)
    'lr': 1e-3,
    'l2': 1e-5,
    'use_bn': True,
    'batch_size': 64
}

# START
hist_light = train_snippet(cfg_light)

In [ ]:
# === CONFIG 3: HIGH REGULARIZATION (Più robustezza) ===
cfg_reg = {
    'name': 'LSTM_HighReg_Run',
    'seq_len': 10,       # Window invariata
    'units1': 256,       # Neuroni come la Best
    'units2': 128,
    'dense_units': 128,
    'dropout': 0.4,      # <-- DROPOUT MOLTO ALTO (40% dei neuroni spenti)
    'lr': 1e-3,
    'l2': 1e-4,          # <-- L2 più aggressiva
    'use_bn': True,
    'batch_size': 64
}

# START
hist_reg = train_snippet(cfg_reg)

In [ ]:
# === CONFIG 4: LONG WINDOW (Time Window = 20) ===
cfg_long = {
    'name': 'LSTM_LongWindow_Run',
    'seq_len': 20,       # <--- CAMBIO FINESTRA TEMPORALE
    'units1': 256,
    'units2': 128,
    'dense_units': 128,
    'dropout': 0.25,     # Leggermente più alto per gestire più dati
    'lr': 1e-3,
    'l2': 1e-5,
    'use_bn': True,
    'batch_size': 64
}

# START
hist_long = train_snippet(cfg_long)

In [ ]:
# === RACCOLTA RISULTATI PER IL REPORT ===
import pandas as pd

# Creiamo una lista con i risultati (assumendo che tu abbia le variabili hist_*)
# Nota: min(hist.history['val_loss']) prende il miglior risultato durante il training

results_data = []

# Funzione helper per estrarre i dati
def extract_res(hist, cfg):
    # Trova l'epoca migliore (quella con val_loss più bassa)
    best_epoch_idx = np.argmin(hist.history['val_loss'])
    return {
        'Model Name': cfg['name'],
        'Window Size': cfg['seq_len'],
        'Units (L1/L2)': f"{cfg['units1']}/{cfg['units2']}",
        'Dropout': cfg['dropout'],
        'Best Val MSE': hist.history['val_loss'][best_epoch_idx],
        'Best Val MAE': hist.history['val_mae'][best_epoch_idx]
        # Nota: Il Test MSE lo hai stampato nei log, qui usiamo Val per confronto rapido
    }

# Aggiungi solo le run che hai effettivamente lanciato
try: results_data.append(extract_res(hist_best, cfg_best))
except: pass
try: results_data.append(extract_res(hist_light, cfg_light))
except: pass
try: results_data.append(extract_res(hist_reg, cfg_reg))
except: pass
try: results_data.append(extract_res(hist_long, cfg_long))
except: pass

# Crea DataFrame
df_results = pd.DataFrame(results_data)

# Ordina dal migliore al peggiore
df_results = df_results.sort_values(by='Best Val MSE', ascending=True)

print("\n" + "="*60)
print("TABLE 1: HYPERPARAMETER TUNING RESULTS")
print("="*60)
print(df_results.to_string(index=False))
print("-" * 60)

# Salva CSV per Excel
df_results.to_csv("final_results_step3.csv", index=False)
print("✅ Tabella salvata in 'final_results_step3.csv'")